# Data Exploration: Hindi, Bengali, and Tamil

This notebook explores the handwritten word datasets used in the project.
It reads the manifest built by `src/manifest.py` and examines sample images,
image sizes, word statistics, and the characters used in each script.

In [ ]:
import os
from pathlib import Path

# Move from the notebooks folder up to the main project folder
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working folder:", Path.cwd())

import pandas as pd
from src.utils import load_config

config = load_config()
df = pd.read_csv(config["data"]["manifest_path"], encoding="utf-8", keep_default_na=False)
print(f"Loaded {len(df):,} rows")
df.head()

## 1. Overview
Number of images per language and split.

In [ ]:
splits = config["data"]["splits"]
pd.crosstab(df["language"], df["split"])[splits]

## 2. Sample images
Four random training images per language, to check that images and labels match.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

SEED = config["seed"]
languages = list(config["data"]["languages"])
n_per_language = 4

fig, axes = plt.subplots(len(languages), n_per_language, figsize=(14, 6))

for row, language in enumerate(languages):
    train_rows = df[(df["language"] == language) & (df["split"] == "train")]
    samples = train_rows.sample(n_per_language, random_state=SEED)

    print(f"{language}:")
    for col, (_, sample) in enumerate(samples.iterrows()):
        ax = axes[row, col]
        with Image.open(sample["image_path"]) as img:
            ax.imshow(img, cmap="gray")
        ax.set_title(f"{language} {col + 1}")
        ax.axis("off")
        print(f"  {col + 1}: {sample['label']}")

plt.tight_layout()
plt.show()

## 3. Image sizes
Width, height, and aspect ratio (width ÷ height) for a random sample of
3,000 training images per language. These guide how images will be resized later.

In [ ]:
sample_size = 3000
size_sample = (
    df[df["split"] == "train"]
    .groupby("language")
    .sample(n=sample_size, random_state=SEED)
    .copy()
)

widths, heights, modes = [], [], []
for path in size_sample["image_path"]:
    with Image.open(path) as img:
        widths.append(img.width)
        heights.append(img.height)
        modes.append(img.mode)

size_sample["width"] = widths
size_sample["height"] = heights
size_sample["mode"] = modes
size_sample["aspect_ratio"] = size_sample["width"] / size_sample["height"]

for column in ["height", "width", "aspect_ratio"]:
    print(f"\n{column.upper()}")
    print(size_sample.groupby("language")[column].describe().round(1))

print("\nCOLOR MODE")
print(pd.crosstab(size_sample["language"], size_sample["mode"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
columns = ["height", "width", "aspect_ratio"]

for language in languages:
    subset = size_sample[size_sample["language"] == language]
    for ax, column in zip(axes, columns):
        ax.hist(subset[column], bins=50, alpha=0.5, label=language)

for ax, column in zip(axes, columns):
    ax.set_title(column)
    ax.set_xlabel("pixels" if column != "aspect_ratio" else "width ÷ height")
    ax.set_ylabel("number of images")
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Word statistics
Word length (in Unicode code points), number of unique words, and how many
test words also appear in the training data.

In [ ]:
# Word length in code points
df["n_codepoints"] = df["label"].str.len()
print("WORD LENGTH (code points)")
print(df.groupby("language")["n_codepoints"].describe().round(1))

# Unique words per language
images = df.groupby("language").size()
unique_words = df.groupby("language")["label"].nunique()
print("\nVOCABULARY")
print(pd.DataFrame({
    "images": images,
    "unique_words": unique_words,
    "images_per_word": (images / unique_words).round(1),
}))

# How many test words also appear in training?
print("\nTEST WORDS ALSO SEEN IN TRAINING")
for language in languages:
    lang_rows = df[df["language"] == language]
    train_words = set(lang_rows[lang_rows["split"] == "train"]["label"])
    test_labels = lang_rows[lang_rows["split"] == "test"]["label"]
    seen = test_labels.isin(train_words).mean()
    print(f"  {language:<8} {seen:.1%}")

## 5. Characters
Every Unicode code point used in each language's labels, how often it appears,
and any characters from outside the language's own script.

In [ ]:
from collections import Counter
import unicodedata

SCRIPT_NAME = {"hindi": "DEVANAGARI", "bengali": "BENGALI", "tamil": "TAMIL"}

def char_table(counter):
    """Turn a Counter of characters into a table with Unicode details."""
    rows = []
    for char, count in counter.most_common():
        rows.append({
            "char": char,
            "code": f"U+{ord(char):04X}",
            "name": unicodedata.name(char, "UNKNOWN"),
            "count": count,
        })
    return pd.DataFrame(rows)

os.makedirs("outputs", exist_ok=True)
char_tables = {}

for language in languages:
    counter = Counter()
    for label in df.loc[df["language"] == language, "label"]:
        counter.update(label)

    table = char_table(counter)
    char_tables[language] = table
    table.to_csv(f"outputs/charset_{language}.csv", index=False, encoding="utf-8-sig")

    outside = table[~table["name"].str.startswith(SCRIPT_NAME[language])]
    print(f"\n{language.upper()}: {len(table)} different code points")
    print(f"Characters outside the {SCRIPT_NAME[language].title()} script:")
    print(outside.to_string(index=False) if len(outside) else "  none")

In [ ]:
for language in languages:
    print(f"\n{language.upper()}: 10 rarest characters")
    print(char_tables[language].tail(10).to_string(index=False))

## 6. Unicode normalization
Which labels were changed by NFC normalization, and which characters were involved.

In [ ]:
from src.manifest import READERS, normalize

# Re-read the original (un-normalized) labels
raw_rows = []
for language, lang_cfg in config["data"]["languages"].items():
    reader = READERS[lang_cfg["format"]]
    for split in splits:
        for row in reader(lang_cfg["folder"], split):
            row["language"] = language
            raw_rows.append(row)

raw = pd.DataFrame(raw_rows)
raw["nfc"] = raw["label"].map(normalize)
changed = raw[raw["label"] != raw["nfc"]]

print("Labels changed per language:")
print(changed.groupby("language").size())

for language in languages:
    lang_changed = changed[changed["language"] == language]
    if len(lang_changed) == 0:
        continue

    removed, added = Counter(), Counter()
    for original, normalized in zip(lang_changed["label"], lang_changed["nfc"]):
        removed.update(set(original) - set(normalized))
        added.update(set(normalized) - set(original))

    print(f"\n{language.upper()}")
    print("Code points that disappeared:")
    print(char_table(removed).to_string(index=False))
    print("Code points that appeared:")
    print(char_table(added).to_string(index=False))

## 7. Summary of findings

**Data size.** 312,023 word images: Hindi 95,430, Bengali 113,075, Tamil 103,518.
No missing images and no empty labels.

**Image sizes.** All images are RGB and much larger than needed. Hindi and Bengali
images are about 290 px tall with almost no variation; Tamil heights vary more.
Median aspect ratio: Hindi 2.5, Bengali 2.9, Tamil 5.4.
→ Convert to grayscale, resize to a fixed height keeping the aspect ratio, and pad
widths within each batch, with a maximum width that allows for long Tamil words.

**Word length.** Median code points per word: Hindi 6, Bengali 7, Tamil 9.
Tamil words are longer and their characters are written wider.

**Vocabulary.** About 9,500–13,300 unique words per language, each written about
8–10 times.

**Test words seen in training.** Hindi 100%, Bengali 46.2%, Tamil 63.2%.
→ Report test results separately for seen and unseen words; do not compare
languages using a single overall score.

**Characters.** Hindi 100, Bengali 87, Tamil 71 code points. The hyphen appears in
all three. Bengali correctly uses the shared Devanagari danda (U+0964).
Many rare characters (Vedic accents, Sindhi letters, currency and special signs)
appear only 5–10 times each, likely from one special word per character.
→ Decide in Phase 3 how to handle ultra-rare characters.

**Unicode normalization.** NFC changed 20,007 labels (Bengali 17,196, Hindi 2,488,
Tamil 323). Dotted letters (nukta forms) were split into base letter + nukta, and
two-part vowel signs were joined into single code points. The datasets contained
both forms mixed together.
→ Always normalize labels to NFC (done in `src/manifest.py`).